In [ ]:
import pandas as pd
import pandas_ta as ta
import yfinance as yf

In [ ]:
org_data = yf.download(tickers='^RUI', start='2014-03-11', end='2024-07-10')
org_data

In [ ]:
data = pd.DataFrame(index=org_data.index)
data['Adj Close'] = org_data['Adj Close']['^RUI']
data['Open'] = org_data['Open']['^RUI']
data['Close'] = org_data['Close']['^RUI']
data['High'] = org_data['High']['^RUI']
data['Low'] = org_data['Low']['^RUI']

In [ ]:
from helper.importer import go

fig = go.Figure()
fig.add_trace(go.Candlestick(x=data.index,
                             open=data['Open'], close=data['Close'], high=data['High'],
                             low=data['Low']))
fig.show()



In [ ]:
scaler = ta.ema(data['Close'], length=500, )

In [ ]:
scaler

In [ ]:
n_data = pd.DataFrame(index=data.index)
n_data['Open'] = data['Open'] / scaler
n_data['Close'] = data['Close'] / scaler
n_data['High'] = data['High'] / scaler
n_data['Low'] = data['Low'] / scaler

In [ ]:
fig = go.Figure()
t = n_data
fig.add_trace(
    go.Candlestick(x=t.index, open=t['Open'], close=t['Close'], high=t['High'], low=t['Low']))
fig.show()

In [ ]:
data.columns

In [ ]:
from sklearn.preprocessing import MinMaxScaler

n2_data = pd.DataFrame(index=data.index)
sc1 = MinMaxScaler(feature_range=(-1, 1))
t = sc1.fit_transform(data.reset_index(drop=True))
n2_data['Adj Close'] = t[:, 0]
n2_data['Open'] = t[:, 1]
n2_data['Close'] = t[:, 2]
n2_data['High'] = t[:, 3]
n2_data['Low'] = t[:, 4]


In [ ]:
fig = go.Figure()
t = n2_data
fig.add_trace(
    go.Candlestick(x=t.index, open=t['Open'], close=t['Close'], high=t['High'], low=t['Low']))
fig.show()

In [ ]:
n2_data

In [ ]:
# n2_data.dropna(subset=['max_high_distance', 'min_low_distance'], inplace=True)
# n2_data

In [ ]:
n2_data[n2_data[['max_high_distance', 'min_low_distance']].isna().any(axis='columns')]

In [ ]:
n2_data

In [ ]:
# t = n2_data[['High', 'Low', 'next_low', 'next_high', 'max_high_distance', 'min_low_distance']]
for i in range(0,len(n2_data)-rolling_window):
    window = n2_data.iloc[i:i+rolling_window]
    index = n2_data.index[i]
    max_long_window = window.iloc[:int(n2_data.iloc[i]['max_high_distance'])]
    max_long_drawdown_list = ((max_long_window['High'] - max_long_window['next_low']).clip(lower=0))
    max_long_drawdown = max_long_drawdown_list
    n2_data.loc[index,'max_long_drawdown'] = max_long_drawdown_list.sum()
    n2_data.loc[index,'count_long_drawdown'] = max_long_drawdown_list.count()
    
    max_short_window = window.iloc[:int(n2_data.iloc[i]['min_low_distance'])]
    max_short_drawdown_list = ((max_short_window['next_high'] - max_short_window['Low']).clip(lower=0))
    max_short_drawdown = max_short_drawdown_list
    n2_data.loc[index,'max_short_drawdown'] = max_short_drawdown_list.sum()
    n2_data.loc[index,'count_short_drawdown'] = max_short_drawdown_list.count()
    

In [ ]:
t

In [ ]:
t = n2_data.sort_index().reset_index(drop=True)
t.index.dtype

In [ ]:
n2_data.columns

In [ ]:
n2_data[['High', 'next_low']]

In [ ]:
t = (
    n2_data[['High', 'next_low']]
    .rolling(window=rolling_window)
    .apply(lambda x: x.mean() - x.mean(), raw=False)
)
t

In [ ]:
n2_data['custom_calculation'] = (
    n2_data[['High', 'next_low']]
    .rolling(window=rolling_window)
    .apply(lambda x: x['High'].mean() - x['next_low'].mean(), raw=False)
)

In [ ]:
t = n2_data[['High', 'next_low', 'max_high_distance']].sort_index().reset_index(drop=True).to_numpy()
t.shape

In [ ]:
import numpy as np
(t.rolling(window=rolling_window, method='table')
    .apply(lambda x: (x[0] - x[1]).mean(), engine='numba'))

In [ ]:
n2_data

In [ ]:
from numba import njit


import numpy as np
import pandas as pd
from numba import njit

# Assuming 'n2_data' is your DataFrame, and 'rolling_window' and 'position_max_time' are defined

@njit
def compute_drawdown(high_vals, next_low_vals, max_high_distance_vals):
    # Ensure the slicing works as expected with NumPy arrays
    max_distance = int(max_high_distance_vals[0])  # Assuming max_high_distance is at index 2
    high_sub = high_vals[:max_distance]
    next_low_sub = next_low_vals[:max_distance]
    
    # Calculate the drawdown, ensure only positive values are kept
    drawdown = np.maximum(high_sub - next_low_sub, 0)
    return np.sum(drawdown)

# Apply rolling window with numba optimization
n2_data['max_long_drawdown'] = (
    n2_data[['High', 'next_low', 'max_high_distance']]
    .rolling(window=rolling_window, min_periods=rolling_window, method='table')
    .apply(lambda x: print(x), 
           engine='numba', raw=True)
    .shift(-position_max_time)  # Adjust the shift as necessary
)


In [ ]:
for i in range(len(n2_data)-rolling_window):
    window = n2_data.iloc[i:i+rolling_window][['High', 'Low', 'next_high', 'next_low', 'max_high_distance', 'min_low_distance']]
    max_high_distance = int(n2_data.iloc[i]['max_high_distance'])
    min_low_distance = int(n2_data.iloc[i]['min_low_distance'])
    long_drawdowns = (window.iloc[i:i+max_high_distance]['High'] - window.iloc[i:i+max_high_distance]['next_low']).clip(lower=0)
    short_drawdowns = (window.iloc[i:i+max_high_distance]['next_high'] - window.iloc[i:i+max_high_distance]['Low']).clip(lower=0)
    n2_data.iloc[i]['max_long_drawdown'] = long_drawdowns.sum()    
    n2_data.iloc[i]['count_long_drawdown'] = long_drawdowns.count()      
    n2_data.iloc[i]['max_short_drawdown'] = short_drawdowns.sum()
    n2_data.iloc[i]['count_short_drawdown'] = short_drawdowns.count()

In [ ]:
# # todo: check direction of rolling window indexes as distance
# n2_data['max_long_drawdown'] = (
#     n2_data[['High', 'next_low', 'max_high_distance']]
#     .rolling(window=rolling_window, min_periods=rolling_window, method='table').apply(
#         lambda x: (x['High'].values[:int(x['max_high_distance'].iloc[0])] -
#                    x['next_low'].values[:int(x['max_high_distance'].iloc[0])]).clip(min=0))
#     # .clip(lower=0)  # Only keep positive differences
#     .sum()
#     .shift(-position_max_time))
# n2_data['max_short_drawdown'] = (
#     n2_data[['Low', 'next_high', 'min_low_distance']]
#     .rolling(window=rolling_window, min_periods=rolling_window, method='table').apply(
#         lambda x: (x['Low'].values[:int(x['min_low_distance'].iloc[0])] -
#                    x['next_high'].values[:int(x['min_low_distance'].iloc[0])]).clip(min=0))
#         # lambda x: x['Low'][: int(x['min_low_distance'][0])] -
#         #           x['next_high'][: int(x['min_low_distance'][0])])
#     # .clip(lower=0)  # Only keep positive differences
#     .sum()
#     .shift(-position_max_time))
# n2_data['count_long_drawdown'] = (
#     n2_data[['High', 'next_low', 'max_high_distance']]
#     .rolling(window=rolling_window, min_periods=rolling_window, method='table').apply(
#         lambda x: (x['High'].values[:int(x['max_high_distance'].iloc[0])] -
#                    x['next_low'].values[:int(x['max_high_distance'].iloc[0])]).clip(min=0))
#     # .clip(lower=0)  # Only keep positive differences
#     .count()
#     .shift(-position_max_time))
# n2_data['count_short_drawdown'] = (
#     n2_data[['Low', 'next_high', 'min_low_distance']]
#     .rolling(window=rolling_window, min_periods=rolling_window, method='table').apply(
#         lambda x: (x['Low'].values[:int(x['min_low_distance'].iloc[0])] -
#                    x['next_high'].values[:int(x['min_low_distance'].iloc[0])]).clip(min=0))
#         # lambda x: x['Low'][: int(x['min_low_distance'][0])] -
#         #           x['next_high'][: int(x['min_low_distance'][0])])
#     # .clip(lower=0)  # Only keep positive differences
#     .count()
#     .shift(-position_max_time))

In [ ]:
# if the count_x_drawdown / position_max_time = 0.1 relative_x_drawdown = 0.9 max_x_drawdown
# if the count_x_drawdown / position_max_time = 0.8 relative_x_drawdown = 0.2 max_x_drawdown
n2_data['relative_long_drawdown'] = (1 - n2_data['count_long_drawdown'] / position_max_time) * n2_data[
    'max_long_drawdown']
n2_data['relative_short_drawdown'] = (1 - n2_data['count_short_drawdown'] / position_max_time) * n2_data[
    'max_short_drawdown']
n2_data['effective_long_power'] = n2_data['weighted_max_long_profit'] - n2_data['relative_long_drawdown']
n2_data['effective_short_power'] = n2_data['weighted_max_short_profit'] - n2_data['relative_short_drawdown']

In [ ]:
from plotly.subplots import make_subplots

t = n2_data
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    row_heights=[0.3, 0.7, 0.3],  # Adjust height proportions if needed
    vertical_spacing=0.01  # Spacing between the charts
)
fig.add_trace(
    go.Candlestick(x=t.index, open=t['Open'], close=t['Close'], high=t['High'], low=t['Low']),
    row=3, col=1)
fig.add_trace(
    go.Scatter(x=t.index, y=t['max_high'], mode='lines', line=dict(color='blue', width=1), name='Long Top'),
    row=3, col=1)
fig.add_trace(
    go.Scatter(x=t.index, y=t['min_low'], mode='lines', line=dict(color='red', width=1), name='Short Top'),
    row=3, col=1)
fig.add_trace(
    go.Scatter(x=t.index, y=t['max_high_distance'], mode='lines', line=dict(color='blue', width=1),
               name='Long Top Distance'),
    row=1, col=1)
fig.add_trace(
    go.Scatter(x=t.index, y=t['min_low_distance'], mode='lines', line=dict(color='red', width=1),
               name='Short Top Distance'),
    row=1, col=1)
fig.add_trace(
    go.Scatter(x=t.index, y=t['max_high_distance'] - t['min_low_distance'], mode='lines',
               line=dict(color='black', width=1), name='Short Top Distance'),
    row=1, col=1)
fig.add_trace(
    go.Scatter(x=t.index, y=(t['max_high'] - t['next_high']) / t['max_high_distance'], mode='lines',
               line=dict(color='blue', width=1), name='Long Power'),
    row=2, col=1)
fig.add_trace(
    go.Scatter(x=t.index, y=(t['next_low'] - t['min_low']) / t['min_low_distance'], mode='lines',
               line=dict(color='red', width=1), name='Short Power'),
    row=2, col=1)
fig.show()